# Finetune Rejection Ensemble Evaluation

Minimal notebook for loading the finetuned hard cascade and measuring validation/test accuracy.

In [ ]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader

TRAINING_ROOT = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training")
if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

from DMTimeShardDataset import DMTimeShardDataset
from training import label_encoding
from moe.train_joint_ensemble import build_joint_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
def get_snr_bins(metadata_path, labels_path, save_pulsars_only):
    """
    Group dataset indices into integer SNR bins based on rounded SNR values.

    Args:
        metadata_path (str): Path to the metadata .npy file.
        labels_path (str): Path to the labels .npy file (used for validation only).

    Returns:
        dict[int | None, list[int]]: Mapping from rounded SNR value to the list of
            dataset indices that fall into that bin. Non-finite SNR values are stored 
            under the ``None`` key.
    """
    if not os.path.exists(metadata_path):
        raise FileNotFoundError(f"Metadata file not found at {metadata_path}")
    if not os.path.exists(labels_path):
        raise FileNotFoundError(f"Labels file not found at {labels_path}")

    metadata = np.load(metadata_path)
    labels = np.load(labels_path)
    if metadata.ndim != 2 or metadata.shape[1] < 1:
        raise ValueError("Expected metadata with shape (N, >=1) where column 0 stores SNR values.")
    if metadata.shape[0] != labels.shape[0]:
        raise ValueError(
            f"Label count ({labels.shape[0]}) does not match metadata count ({metadata.shape[0]})."
        )
        
    print("metadata length: ", len(metadata), "labels length: ", len(labels))
    if metadata.size == 0:
        return {}
        
    snr_values = metadata[:, 0]
    snr_bins = defaultdict(list)
    
    for idx, snr_value in enumerate(snr_values):
        is_finite = np.isfinite(snr_value)
        rounded_bin = int(np.rint(float(snr_value))) if is_finite else None
        
        if save_pulsars_only and labels[idx] != "Pulse":
            continue
        
        snr_bins[rounded_bin].append(idx)
        
    finite_keys = sorted([key for key in snr_bins.keys() if key is not None])
    ordered_bins = {key: snr_bins[key] for key in finite_keys}
    if None in snr_bins:
        ordered_bins[None] = snr_bins[None]
        
    print(
        f"Created {len(ordered_bins)} SNR bins from {len(snr_values)} samples."
    )
    if None in ordered_bins:
        print(f"  - {len(ordered_bins[None])} samples contain non-finite SNR values and are stored under None.")
    return ordered_bins

In [ ]:
def evaluate_model_for_snrs(
    model,
    snr_bins,
    data_path,
    labels_path,
    batch_size=512,
    num_workers=8,
    use_freq_time=True,
    split="val",
    device=None,
    positive_label=1,
    zero_division=0.0,
):
    """
    Evaluate ``model`` on the selected dataset split for each SNR bin and report detailed metrics.

    Args:
        model: Trained classifier providing ``model(batch_dict)`` predictions.
        snr_bins (dict[int | None, list[int]]): Mapping from rounded SNR value to
            *global* dataset indices (see :func:`get_snr_bins`).
        data_path (str | Path): Directory that holds the DM-time dataset shards.
        labels_path (str | Path): Absolute path to the ``*_labels.npy`` file (used to infer the prefix).
        batch_size (int, optional): Mini-batch size for evaluation DataLoaders.
        num_workers (int, optional): Worker count for the evaluation DataLoaders.
        use_freq_time (bool, optional): Whether to load freq-time tensors alongside DM-time.
        split (str, optional): Dataset split to evaluate ("train", "val" or "test"). Defaults to "test".
        device (torch.device | str | None, optional): Device to execute the model on.
        positive_label (int, optional): Label considered the positive class when computing precision/recall/F1.
        zero_division (float, optional): Value to use when a metric has a zero denominator.

    Returns:
        dict: Summary containing per-bin metrics (accuracy, precision, recall, F1) and overall metrics.
    """

    def _infer_prefix(labels_path_: str) -> str:
        labels_name = os.path.basename(labels_path_)
        suffixes = [
            "_DM_time_dataset_realbased_labels_test.npy",
            "_DM_time_dataset_realbased_labels_train.npy",
            "_DM_time_dataset_realbased_labels_val.npy",
            "_DM_time_dataset_realbased_labels_joined.npy",
        ]
        for suffix in suffixes:
            if labels_name.endswith(suffix):
                return labels_name[: -len(suffix)]
        return Path(labels_name).stem

    def _compute_metrics(tp, fp, fn, tn):
        total = tp + fp + fn + tn
        accuracy = (tp + tn) / total if total else float("nan")
        precision = tp / (tp + fp) if (tp + fp) else zero_division
        recall = tp / (tp + fn) if (tp + fn) else zero_division
        if precision + recall > 0:
            f1 = 2 * precision * recall / (precision + recall)
        else:
            f1 = zero_division
        return {
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "count": total,
        }

    def _format_metric(value):
        return f"{value:.4f}" if math.isfinite(value) else "   nan"

    data_root = Path(data_path)
    if data_root.is_file():
        data_root = data_root.parent
    if not data_root.exists():
        data_root = Path(labels_path).resolve().parent

    dataset_cfg = {
        "output_dir": str(data_root),
        "prefix": _infer_prefix(str(labels_path)),
    }

    print(
        f"Loading dataset from {dataset_cfg['output_dir']} (prefix={dataset_cfg['prefix']}, split={split})"
    )
    full_dataset = DMTimeShardDataset(
        dataset_cfg,
        use_freq_time=use_freq_time,
        split=split,
    )
    full_dataset.labels = label_encoding(full_dataset.labels.astype(object))

    dataset_len = len(full_dataset)
    print(f"Dataset split '{split}' contains {dataset_len} samples")

    metadata = None
    metadata_path = Path(labels_path).with_name(
        Path(labels_path).name.replace("labels", "metadata")
    )
    if metadata_path.exists():
        metadata = np.load(metadata_path)

    if device is None:
        first_param = next(model.parameters(), None)
        if first_param is not None:
            target_device = first_param.device
        else:
            target_device = torch.device("cpu")
    else:
        target_device = device if isinstance(device, torch.device) else torch.device(device)

    model.eval()

    def _move_batch_to_device(batch):
        return {k: v.to(target_device) if torch.is_tensor(v) else v for k, v in batch.items()}

    bin_results = {}
    overall_counts = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}

    def _bin_sort_key(item):
        key, _ = item
        return (float("inf"), 0) if key is None else (0, key)
    
    wrong_samples = []

    with torch.no_grad():
        for snr_value, indices in sorted(snr_bins.items(), key=_bin_sort_key):
            if not indices:
                metrics = {
                    "accuracy": float("nan"),
                    "precision": zero_division,
                    "recall": zero_division,
                    "f1": zero_division,
                    "count": 0,
                }
                bin_results[snr_value] = {**metrics, "tp": 0, "fp": 0, "fn": 0, "tn": 0}
                continue

            valid_indices = [idx for idx in indices if idx < dataset_len]
            if not valid_indices:
                print(
                    f"Skipping SNR bin {snr_value} because all {len(indices)} indices exceed the '{split}' split length."
                )
                bin_results[snr_value] = {
                    "accuracy": float("nan"),
                    "precision": zero_division,
                    "recall": zero_division,
                    "f1": zero_division,
                    "count": 0,
                    "tp": 0,
                    "fp": 0,
                    "fn": 0,
                    "tn": 0,
                }
                continue

            if len(valid_indices) < len(indices):
                print(
                    f"Warning: {len(indices) - len(valid_indices)} indices in SNR bin {snr_value} were clipped to the '{split}' split length."
                )

            subset = torch.utils.data.Subset(full_dataset, valid_indices)
            loader = DataLoader(
                subset, batch_size=batch_size, shuffle=False, num_workers=num_workers
            )

            counts = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}
            batch_offset = 0
            for batch in loader:
                batch_on_device = _move_batch_to_device(batch)
                labels = batch_on_device["label"]
                outputs = model.forward(batch_on_device)
                predictions = outputs.argmax(dim=1)

                batch_size_local = labels.shape[0]
                batch_indices = valid_indices[batch_offset:batch_offset + batch_size_local]
                batch_offset += batch_size_local
                
                # Find indices where predictions don't match labels
                wrong_mask = labels != predictions
                if wrong_mask.any():
                    wrong_mask_cpu = wrong_mask.detach().cpu().numpy()
                    wrong_indices = [
                        batch_indices[i]
                        for i in range(len(batch_indices))
                        if wrong_mask_cpu[i]
                    ]

                    # save wrongly classified samples with SNR bucket and label info
                    wrong_batch = {}
                    for key, value in batch.items():
                        if torch.is_tensor(value):
                            wrong_batch[key] = value[wrong_mask.cpu()].cpu()
                        else:
                            wrong_batch[key] = [value[i] for i in range(len(value)) if wrong_mask.cpu()[i]]
                    
                    # add SNR bucket information
                    wrong_batch['snr_bucket'] = snr_value
                    wrong_batch['true_labels'] = labels[wrong_mask].cpu()
                    wrong_batch['predictions'] = predictions[wrong_mask].cpu()
                    wrong_batch['sample_indices'] = wrong_indices
                    if metadata is not None:
                        wrong_batch['metadata_rows'] = np.asarray([metadata[idx] for idx in wrong_indices])
                    
                    wrong_samples.append(wrong_batch)

                positive_pred = predictions == positive_label
                positive_true = labels == positive_label

                tp_batch = torch.logical_and(positive_pred, positive_true).sum().item()
                fp_batch = torch.logical_and(positive_pred, ~positive_true).sum().item()
                fn_batch = torch.logical_and(~positive_pred, positive_true).sum().item()
                tn_batch = torch.logical_and(~positive_pred, ~positive_true).sum().item()

                counts["tp"] += tp_batch
                counts["fp"] += fp_batch
                counts["fn"] += fn_batch
                counts["tn"] += tn_batch

            metrics = _compute_metrics(**counts)
            bin_results[snr_value] = {**metrics, **counts}
            overall_counts["tp"] += counts["tp"]
            overall_counts["fp"] += counts["fp"]
            overall_counts["fn"] += counts["fn"]
            overall_counts["tn"] += counts["tn"]

            label = "non-finite" if snr_value is None else f"SNR={snr_value}"
            print(
                f"{label:<12}: samples={metrics['count']:5d}, "
                f"acc={_format_metric(metrics['accuracy'])} "
                f"prec={_format_metric(metrics['precision'])} "
                f"rec={_format_metric(metrics['recall'])} "
                f"f1={_format_metric(metrics['f1'])}"
            )

    overall_metrics = _compute_metrics(**overall_counts)
    print("-" * 60)
    print(
        f"Overall across {overall_metrics['count']} samples -> "
        f"acc={_format_metric(overall_metrics['accuracy'])}, "
        f"prec={_format_metric(overall_metrics['precision'])}, "
        f"rec={_format_metric(overall_metrics['recall'])}, "
        f"f1={_format_metric(overall_metrics['f1'])}"
    )

    return ({
        "per_bin": bin_results,
        "overall": {**overall_metrics, **overall_counts},
        "overall_accuracy": overall_metrics["accuracy"],
    }, wrong_samples)

In [ ]:
def plot_snr_curves(snr_evaluations, SNR_end=None, SNR_start=None, y_min=0.0, include_non_finite=False):
    """
    Plot accuracy, precision, recall, and F1-score curves as a function of SNR.

    Args:
        snr_evaluations (list[dict]): Output list produced by `evaluate_model_for_snrs`.
        SNR_end (float, optional): Maximum SNR value to plot.
        SNR_start (float, optional): Minimum SNR value to plot.
        y_min (float, optional): Minimum y-axis value to make high accuracies visible.
        include_non_finite (bool, optional): Whether to include the `None` bin (non-finite
            SNR values) at the leftmost position. Defaults to False.

    Returns:
        matplotlib.axes.Axes: Axis containing the rendered plot.
    """
    
    def _sort_key(item):
        key, _ = item
        return (float("inf"), 0) if key is None else (0, key)
    
    for snr_evaluation in snr_evaluations:
        
        if not isinstance(snr_evaluation, dict) or "per_bin" not in snr_evaluation:
            raise ValueError("snr_evaluation must contain a 'per_bin' entry.")

        per_bin = snr_evaluation["per_bin"]
        if not per_bin:
            raise ValueError("snr_evaluation['per_bin'] is empty; nothing to plot.")

        if SNR_start is not None or (SNR_end is not None and SNR_end > 0):
            new_per_bin = dict()
            for key in per_bin.keys():
                if key is None:
                    new_per_bin[key] = per_bin[key]
                else:
                    if SNR_start is not None and key < SNR_start:
                        continue
                    if SNR_end is not None and SNR_end > 0 and key > SNR_end:
                        continue
                    new_per_bin[key] = per_bin[key]
            per_bin = new_per_bin
        
        metric_names = ["accuracy", "precision", "recall", "f1"]
        style_cycle = {
            "accuracy": {"marker": "o", "linestyle": "-", "color": "#1f77b4"},
            "precision": {"marker": "s", "linestyle": "--", "color": "#ff7f0e"},
            "recall": {"marker": "D", "linestyle": "-.", "color": "#2ca02c"},
            "f1": {"marker": "^", "linestyle": ":", "color": "#d62728"},
        }


        snr_values = []
        metrics_by_name = {name: [] for name in metric_names}
        has_non_finite = False

        for snr_value, metrics in sorted(per_bin.items(), key=_sort_key):
            if snr_value is None and not include_non_finite:
                has_non_finite = True
                continue
            snr_label = -1 if snr_value is None else snr_value
            snr_values.append(snr_label)
            for name in metric_names:
                metrics_by_name[name].append(metrics.get(name, float("nan")))

        if not snr_values:
            raise ValueError("No SNR bins left to plot. Enable include_non_finite to show None bin.")

        fig, ax = plt.subplots(figsize=(7, 5))
        for name in metric_names:
            ax.plot(
                snr_values,
                metrics_by_name[name],
                label=name.capitalize(),
                **style_cycle.get(name, {}),
            )

    ax.set_xlabel("SNR")
    ax.set_ylabel("Score")
    ax.set_title("Per-SNR Detection Metrics")
    ax.set_ylim(y_min, 1.005)
    ax.set_xlim(min(snr_values), max(snr_values))
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right")


    plt.tight_layout()
    return ax

In [ ]:
DATASET_CFG = {
    "output_dir": "/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs",
    "prefix": "B0531+21_59000_48386",
}

FINETUNE_REJECTION_DIR = TRAINING_ROOT / "final_checkpoints" / "finetune_checkpoints"

R1_THRESHOLD = 0.542446
R2_THRESHOLD = 0.40963032841682434

FINETUNE_JOINT_CONFIG = {
    "dataset": DATASET_CFG,
    "model": {
        "temperature": 1.0,
        "f_small": {
            "model_name": "DM_time_binary_classificator_241002_3_GAP",
            "resolution": 256,
            "mode": "dmt",
            "dropout": False,
            "checkpoint": str(FINETUNE_REJECTION_DIR / "prot-DM_time_binary_classificator_241002_3_GAP_finetune-004-0.838-0.813.pth"),
        },
        "f_mid": {
            "model_name": "DM_time_binary_classificator_241002_5_GAP",
            "resolution": 256,
            "mode": "ft",
            "dropout": False,
            "checkpoint": str(FINETUNE_REJECTION_DIR / "prot-DM_time_binary_classificator_241002_5_GAP_finetune-019-0.989-0.993.pth"),
        },
        "f_large": {
            "model_name": "DM_time_binary_classificator_resnet18",
            "resolution": 256,
            "mode": "dmft",
            "dropout": False,
            "checkpoint": str(FINETUNE_REJECTION_DIR / "prot-DM_time_binary_classificator_resnet18_finetune-010-0.999-0.993.pth"),
        },
        "r1": {
            "model_name": "conv_mlp",
            "cnn_channels": 64,
            "extra_conv": False,
            "pool_size": 7,
            "hidden_dim": 64,
            "dropout": 0.0,
            "checkpoint": str(FINETUNE_REJECTION_DIR / "prot-run_embedding_r1_conv_mlp_lr1.05e-05_wd0.00e+00_drop0.0_channels64_extraFalse_pool7_hidden64_worker3_trial0-030-0.688-0.618.pth"),
        },
        "r2": {
            "model_name": "conv_mlp",
            "cnn_channels": 64,
            "extra_conv": True,
            "pool_size": 7,
            "hidden_dim": 128,
            "dropout": 0.2,
            "checkpoint": str(FINETUNE_REJECTION_DIR / "prot-run_embedding_r2_conv_mlp_lr4.51e-05_wd0.00e+00_drop0.2_channels64_extraTrue_pool7_hidden128_worker0_trial0-017-0.800-0.834.pth"),
        },
    },
}

print(f"R1_THRESHOLD={R1_THRESHOLD:.6f}")
print(f"R2_THRESHOLD={R2_THRESHOLD:.17f}")


In [ ]:
class HardThresholdJointCascade(torch.nn.Module):
    def __init__(self, cascade, threshold_r1, threshold_r2):
        super().__init__()
        self.cascade = cascade
        self.threshold_r1 = float(threshold_r1)
        self.threshold_r2 = float(threshold_r2)
        self.reset_counts()

    def reset_counts(self):
        self.route_counts = {"small": 0, "mid": 0, "large": 0, "total": 0}

    def forward(self, batch):
        outputs = self.cascade.forward_hard_aux(
            batch,
            threshold_r1=self.threshold_r1,
            threshold_r2=self.threshold_r2,
        )
        selected = outputs["selected_expert"].detach().cpu()
        counts = torch.bincount(selected, minlength=3).tolist()
        self.route_counts["small"] += int(counts[0])
        self.route_counts["mid"] += int(counts[1])
        self.route_counts["large"] += int(counts[2])
        self.route_counts["total"] += int(selected.numel())
        return outputs["log_probs"]

    def print_counts(self):
        counts = self.route_counts
        print("Hard finetune routing:", counts)
        if counts["total"]:
            print(f"small route quote: {counts['small'] / counts['total']:.4f}")
            print(f"mid route quote:   {counts['mid'] / counts['total']:.4f}")
            print(f"large route quote: {counts['large'] / counts['total']:.4f}")


finetune_joint_moe = build_joint_model(FINETUNE_JOINT_CONFIG, device=device).eval()
finetune_rejection_ensemble = HardThresholdJointCascade(
    finetune_joint_moe,
    threshold_r1=R1_THRESHOLD,
    threshold_r2=R2_THRESHOLD,
).to(device)
finetune_rejection_ensemble.eval()

print(f"Loaded finetune rejection ensemble on {device}")


In [ ]:
def evaluate_full_split_accuracy(
    model,
    dataset_cfg,
    split,
    batch_size=512,
    num_workers=0,
    device=None,
):
    dataset = DMTimeShardDataset(dataset_cfg, use_freq_time=True, split=split)
    dataset.labels = label_encoding(dataset.labels.astype(object))
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

    target_device = device if isinstance(device, torch.device) else torch.device(device or "cpu")
    model.eval()
    if hasattr(model, "reset_counts"):
        model.reset_counts()

    correct = 0
    total = 0
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(target_device) if torch.is_tensor(v) else v for k, v in batch.items()}
            labels = batch["label"]
            predictions = model(batch).argmax(dim=1)
            correct += int((predictions == labels).sum().item())
            total += int(labels.numel())

    accuracy = correct / total if total else float("nan")
    print(f"{split} accuracy: {accuracy:.12f} ({correct}/{total})")
    if hasattr(model, "print_counts"):
        model.print_counts()
    return {
        "split": split,
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "route_counts": dict(getattr(model, "route_counts", {})),
    }


In [ ]:
finetune_val_accuracy = evaluate_full_split_accuracy(
    finetune_rejection_ensemble,
    DATASET_CFG,
    split="val",
    batch_size=512,
    num_workers=0,
    device=device,
)


In [ ]:
finetune_test_accuracy = evaluate_full_split_accuracy(
    finetune_rejection_ensemble,
    DATASET_CFG,
    split="test",
    batch_size=512,
    num_workers=0,
    device=device,
)


In [ ]:
val_metadata_path = Path(DATASET_CFG["output_dir"]) / f"{DATASET_CFG['prefix']}_DM_time_dataset_realbased_metadata_val.npy"
val_labels_path = Path(DATASET_CFG["output_dir"]) / f"{DATASET_CFG['prefix']}_DM_time_dataset_realbased_labels_val.npy"
finetune_snr_bins_val = get_snr_bins(str(val_metadata_path), str(val_labels_path), save_pulsars_only=False)

snr_evaluation_finetune_val, wrong_samples_finetune_val = evaluate_model_for_snrs(
    finetune_rejection_ensemble,
    finetune_snr_bins_val,
    data_path=Path(DATASET_CFG["output_dir"]),
    labels_path=val_labels_path,
    batch_size=512,
    num_workers=0,
    use_freq_time=True,
    split="val",
    device=device,
)

plot_snr_curves(
    [snr_evaluation_finetune_val],
    SNR_start=1,
    SNR_end=14,
    y_min=0.0,
)
plt.show()


In [ ]:
test_metadata_path = Path(DATASET_CFG["output_dir"]) / f"{DATASET_CFG['prefix']}_DM_time_dataset_realbased_metadata_test.npy"
test_labels_path = Path(DATASET_CFG["output_dir"]) / f"{DATASET_CFG['prefix']}_DM_time_dataset_realbased_labels_test.npy"
finetune_snr_bins_test = get_snr_bins(str(test_metadata_path), str(test_labels_path), save_pulsars_only=False)

snr_evaluation_finetune_test, wrong_samples_finetune_test = evaluate_model_for_snrs(
    finetune_rejection_ensemble,
    finetune_snr_bins_test,
    data_path=Path(DATASET_CFG["output_dir"]),
    labels_path=test_labels_path,
    batch_size=512,
    num_workers=0,
    use_freq_time=True,
    split="test",
    device=device,
)

plot_snr_curves(
    [snr_evaluation_finetune_test],
    SNR_start=1,
    SNR_end=14,
    y_min=0.0,
)
plt.show()


# Measured Results

Using final checkpoints from `single_pulse_classifier_training/final_checkpoints/finetune_checkpoints` and hard thresholds:

- `R1_THRESHOLD = 0.542446`
- `R2_THRESHOLD = 0.40963032841682434`

Measured accuracies:

- Val accuracy: `0.859340122768` (`147834 / 172032`)
- Test accuracy: `0.870577939540` (`150213 / 172544`)

Test routing counts:

- `f_small`: `121218`
- `f_mid`: `36168`
- `f_large`: `15158`
